# Baby Step 10 — Integrated Application, Permissions and Dry-Run Automation

This notebook validates the application layer that makes the vault usable as a read-only operating system. It executes the same deterministic health check as the command-line application.


## Control question

Can the complete exo-brain be exposed as an application while preserving explicit permission boundaries and human authority?


In [ ]:
from pathlib import Path
import os, json, subprocess, sys
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display=print
VAULT_NAME="Alejandro-Reynoso-Investment-Banking-Vault"
env=os.environ.get("VAULT_PATH")
candidates=([Path(env)] if env else [])+[Path.cwd()/VAULT_NAME,Path("/content/drive/MyDrive")/VAULT_NAME,Path("/workspace/scratch/9ba1ff46ede5")/VAULT_NAME]
VAULT=next((p for p in candidates if p.exists()),None)
if VAULT is None: raise FileNotFoundError("Set VAULT_PATH or mount the vault.")
APP=VAULT/"Application/investment_banking_os_app.py"
print("Vault:",VAULT)


## 1. Validate integration files


In [ ]:
required=[APP,VAULT/"Application/system_manifest.json",VAULT/"Application/permission_matrix.json",VAULT/"Application/automation_schedule.json",VAULT/"Application/mcp_vault_config.example.json"]
missing=[str(p) for p in required if not p.exists()]
assert not missing,missing
print("Validated",len(required),"application files")


## 2. Execute the application health check


In [ ]:
result=subprocess.run([sys.executable,str(APP),"--health-check","--json"],capture_output=True,text=True,check=True)
health=json.loads(result.stdout)
assert health["status"]=="PASS"
display(health)


## 3. Verify expected system counts


In [ ]:
expected={"companies":103,"counterparties":19,"sources":21,"claims":42,"notebooks":11}
assert health["counts"]==expected
pd.DataFrame([health["counts"],expected],index=["Observed","Expected"])


## 4. Inspect the permission matrix


In [ ]:
permissions=pd.read_json(VAULT/"Application/permission_matrix.json")
display(permissions)
denied=set(permissions.query('default=="DENY"')["capability"])
assert {"Write vault files","Send email or messages","Contact counterparties","Execute transactions"}.issubset(denied)


## 5. Inspect dry-run automation


In [ ]:
schedule=pd.read_json(VAULT/"Application/automation_schedule.json")
assert len(schedule)==6 and (schedule["mode"]=="DRY_RUN").all()
display(schedule[["job_id","cadence","task","mode","human_gate"]])


## 6. Verify the application snapshot


In [ ]:
snapshot_result=subprocess.run([sys.executable,str(APP),"--snapshot","--json"],capture_output=True,text=True,check=True)
snapshot=json.loads(snapshot_result.stdout)
assert len(snapshot["top_opportunities"])==5 and snapshot["claim_summary"]=={"total":42,"stale":2}
display(pd.DataFrame(snapshot["top_opportunities"]))


## 7. Verify the operating-playbook layer


In [ ]:
playbooks=list((VAULT/"Playbooks").glob("*Playbook.md"))
assert len(playbooks)==5
print([p.name for p in sorted(playbooks)])


## 8. Test the prototype boundary


In [ ]:
boundary={"read_only":health["mode"]=="read-only","external_action_disabled":health["external_action_enabled"] is False,"all_jobs_dry_run":(schedule["mode"]=="DRY_RUN").all(),"synthetic":health["checks"]["synthetic_boundary"]}
assert all(bool(v) for v in boundary.values())
print(boundary)


## 9. Productionization backlog


In [ ]:
backlog=pd.DataFrame([["Identity","Role-based access and strong authentication"],["Security","Secrets management, encryption and isolated execution"],["Data","Approved connectors, entitlements and retention"],["Governance","Model risk, legal, compliance and supervisory review"],["Reliability","Tests, monitoring, disaster recovery and immutable audit"],["Operations","Controlled scheduler and change-management process"]],columns=["Workstream","Requirement"])
display(backlog)


## 10. Decision engine


In [ ]:
checks={"health_pass":health["status"]=="PASS","counts_match":health["counts"]==expected,"permissions_safe":len(denied)>=4,"automation_dry_run":(schedule["mode"]=="DRY_RUN").all(),"production_ready":False}
assert all(v for k,v in checks.items() if k!="production_ready") and not checks["production_ready"]
recommendation="ACCEPT READ-ONLY PROTOTYPE — PRODUCTIONIZATION REQUIRED"
print(checks); print(recommendation)


## Result

The architecture now has an application boundary: **vault → governed adapters → read-only views → permission gate → dry-run automation → human decision**. It is a complete prototype, not a production deployment.
